In [1]:
# 1. 라이브러리 불러오기 및 오염된 실습 데이터 생성
# 일부러 실무에서 자주 만나는 온갖 오류 데이터(잘못된 문자열, 타입 오류, 말도 안 되는 숫자 등)를 섞어 데이터프레임을 생성합니다.

import pandas as pd
import numpy as np

# 실습용 오류 데이터 생성
data = {
    '사원ID': ['E01', 'E02', 'E03', 'E04', 'E05'],
    '가입일자': ['2025-01-01', '2025.02.15', '2025/03/20', '2025-04-10', '2025-05-25'], # 날짜 포맷 제각각
    '부서': [' 마케팅부 ', '영업부', '회계부', '마케팅', '영업부_확인'], # 공백 및 오탈자
    '나이': [28, -5, 32, 999, 31], # 음수와 극단적인 이상치 포함
    '급여': ['3,500', '4,000', '4,200', '3,800', '5,000'] # 숫자가 아닌 콤마(,)가 포함된 문자열 타입
}

df = pd.DataFrame(data)
print("--- 수정 전 원본 데이터프레임 ---")
print(df)
df.info() # 급여와 가입일자가 object(문자열) 타입인 것을 확인

--- 수정 전 원본 데이터프레임 ---
  사원ID        가입일자      부서   나이     급여
0  E01  2025-01-01   마케팅부    28  3,500
1  E02  2025.02.15     영업부   -5  4,000
2  E03  2025/03/20     회계부   32  4,200
3  E04  2025-04-10     마케팅  999  3,800
4  E05  2025-05-25  영업부_확인   31  5,000
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   사원ID    5 non-null      str  
 1   가입일자    5 non-null      str  
 2   부서      5 non-null      str  
 3   나이      5 non-null      int64
 4   급여      5 non-null      str  
dtypes: int64(1), str(4)
memory usage: 332.0 bytes


In [2]:
# 2. 문자열 데이터 교정 (공백 제거 및 텍스트 수정)
# 부서 이름의 앞뒤 공백을 제거하고, 제각각인 명칭을 하나로 통일합니다.

# Copy본으로 실습 진행
df_clean = df.copy()

# 1. str.strip()으로 문자열 앞뒤의 무의미한 공백 제거
df_clean['부서'] = df_clean['부서'].str.strip()

# 2. str.replace() 또는 replace() 딕셔너리를 활용해 오탈자 및 명칭 통일
# '마케팅' -> '마케팅부', '영업부_확인' -> '영업부'로 교정
corrections = {'마케팅': '마케팅부', '영업부_확인': '영업부'}
df_clean['부서'] = df_clean['부서'].replace(corrections)

print("--- 1단계: 문자열 교정 후 ---")
print(df_clean['부서'])

--- 1단계: 문자열 교정 후 ---
0    마케팅부
1     영업부
2     회계부
3    마케팅부
4     영업부
Name: 부서, dtype: str


In [3]:
# 3. 데이터 타입 변환 및 포맷 교정 (급여, 날짜)
# 연산을 해야 하는 '급여'는 숫자형으로, '가입일자'는 날짜형(Datetime)으로 올바르게 타입을 수정합니다.

# 1. 급여 열에서 콤마(,) 제거 후 정수형(int)으로 변환
df_clean['급여'] = df_clean['급여'].str.replace(',', '').astype(int)

# 2. 다양한 형태로 적힌 날짜 형식을 표준 포맷(YYYY-MM-DD)의 Datetime 타입으로 변환
# format='mixed'를 사용하면 서로 다른 날짜 구분자(., /, -)를 알아서 추론하여 맞춰줍니다.
df_clean['가입일자'] = pd.to_datetime(df_clean['가입일자'], format='mixed')

print("--- 2단계: 타입 변환 후 ---")
df_clean.info()

--- 2단계: 타입 변환 후 ---
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   사원ID    5 non-null      str           
 1   가입일자    5 non-null      datetime64[us]
 2   부서      5 non-null      str           
 3   나이      5 non-null      int64         
 4   급여      5 non-null      int64         
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 332.0 bytes


In [4]:
# 4. 범위 및 수치 이상치 교정 (나이)
# 나이 데이터의 음수(-5)와 말도 안 되는 값(999)을 상식적인 범위 안의 값으로 수정하거나 처리합니다.

# 방법 A: 조건에 맞춰 다른 값으로 직접 대체하기 (예: 음수는 0으로, 100세 이상은 결측치 처리)
df_clean.loc[df_clean['나이'] < 0, '나i'] = 0
df_clean.loc[df_clean['나이'] > 120, '나이'] = np.nan # 999살은 NaN으로 처리 후 fillna 등으로 해결

# 방법 B: 상한선과 하한선을 정해두고 범위를 벗어나는 값을 강제로 맞추기 (Clip 기능)
# 예: 나이의 정상 범위를 20세 ~ 65세로 제한하고 싶을 때
df_clip = df_clean.copy()
df_clip['나이'] = df_clip['나이'].clip(lower=20, upper=65)

print("--- 3단계: 나이 이상치 교정 후 (Clip 적용) ---")
print(df_clip)

--- 3단계: 나이 이상치 교정 후 (Clip 적용) ---
  사원ID       가입일자    부서    나이    급여   나i
0  E01 2025-01-01  마케팅부  28.0  3500  NaN
1  E02 2025-02-15   영업부  20.0  4000  0.0
2  E03 2025-03-20   회계부  32.0  4200  NaN
3  E04 2025-04-10  마케팅부   NaN  3800  NaN
4  E05 2025-05-25   영업부  31.0  5000  NaN


In [5]:
# 5. 파생 변수 생성을 통한 데이터 수정 (연봉 계산)
# 기존에 오염되어 있던 '급여'를 3단계에서 숫자로 정상 교정했기 때문에, 이제 안전하게 수치 연산을 적용하여 새로운 컬럼을 만들 수 있습니다.

# 월급 개념의 급여를 연봉(급여 * 12) 데이터로 가공 및 수정
df_clip['연봉'] = df_clip['급여'] * 12

print("--- 최종 교정 및 가공 완료 데이터 ---")
print(df_clip)

--- 최종 교정 및 가공 완료 데이터 ---
  사원ID       가입일자    부서    나이    급여   나i     연봉
0  E01 2025-01-01  마케팅부  28.0  3500  NaN  42000
1  E02 2025-02-15   영업부  20.0  4000  0.0  48000
2  E03 2025-03-20   회계부  32.0  4200  NaN  50400
3  E04 2025-04-10  마케팅부   NaN  3800  NaN  45600
4  E05 2025-05-25   영업부  31.0  5000  NaN  60000
